In [1]:
import sys
print(sys.executable)

c:\Users\Alive\.conda\envs\eegnet\python.exe


In [2]:
print("hello")

hello


In [3]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\Alive\Documents\eeg-brain-networks-dementia


In [ ]:
import os
os.chdir("..")
print(os.getcwd())

In [1]:
import openneuro as on

on.download(dataset="ds004504", target_dir="data/ds004504",
            include=["participants.tsv", "README", "dataset_description.json"])

🌍 Preparing to download ds004504 …
📥 Checking 5 files, downloading as needed (5 concurrent downloads). 


Output()

✅ Finished downloading ds004504 (downloaded 5 files and 9.6 kB).


In [2]:
import pandas as pd

df = pd.read_csv("data/ds004504/participants.tsv", sep="\t")
print(df.columns.tolist())
print(df.head())

['participant_id', 'Gender', 'Age', 'Group', 'MMSE']
  participant_id Gender  Age Group  MMSE
0        sub-001      F   57     A    16
1        sub-002      F   78     A    22
2        sub-003      M   70     A    14
3        sub-004      F   67     A    20
4        sub-005      M   70     A    22


In [3]:
print(df["Group"].value_counts())
print()
for g in ["A", "F", "C"]:
    print(g, df.loc[df.Group == g, "participant_id"].tolist()[:3], "...")

Group
A    36
C    29
F    23
Name: count, dtype: int64

A ['sub-001', 'sub-002', 'sub-003'] ...
F ['sub-066', 'sub-067', 'sub-068'] ...
C ['sub-037', 'sub-038', 'sub-039'] ...


In [4]:
summary = df.groupby("Group").agg(n=("participant_id", "count"),
                                  age_mean=("Age", "mean"), age_sd=("Age", "std"),
                                  mmse_mean=("MMSE", "mean"), mmse_sd=("MMSE", "std"))
print(summary.round(1))
print(pd.crosstab(df["Group"], df["Gender"]))

        n  age_mean  age_sd  mmse_mean  mmse_sd
Group                                          
A      36      66.4     7.9       17.8      4.5
C      29      67.9     5.4       30.0      0.0
F      23      63.7     8.2       22.2      2.6
Gender   F   M
Group         
A       24  12
C       11  18
F        9  14


In [5]:
from scipy.stats import kruskal

ages = [g["Age"].values for _, g in df.groupby("Group")]
print(kruskal(*ages))

KruskalResult(statistic=np.float64(3.1739703278483433), pvalue=np.float64(0.20454134172136804))


In [6]:
import os
print(os.path.exists("results/tables/cohort_summary.csv"))

False


In [7]:
import os
print(os.getcwd())
print(os.path.exists("results/tables"))

c:\Users\Alive\Documents\eeg-brain-networks-dementia\notebooks
False


In [8]:
import os
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("report", exist_ok=True)
os.makedirs("src", exist_ok=True)
os.makedirs("data", exist_ok=True)
print(os.path.exists("results/tables"))

True


In [9]:
summary = df.groupby("Group").agg(n=("participant_id", "count"),
                                  age_mean=("Age", "mean"), age_sd=("Age", "std"),
                                  mmse_mean=("MMSE", "mean"), mmse_sd=("MMSE", "std"))
summary.round(1).to_csv("results/tables/cohort_summary.csv")
print(os.path.exists("results/tables/cohort_summary.csv"))

True


In [10]:
print(sorted(os.listdir(".")))

['.gitkeep', 'audit.ipynb', 'data', 'report', 'results', 'src']


In [11]:
import os
os.chdir("..")
print(os.getcwd())
print(sorted(os.listdir(".")))

c:\Users\Alive\Documents\eeg-brain-networks-dementia
['.git', '.gitignore', 'LICENSE', 'README.md', 'data', 'notebooks', 'report', 'results', 'src']


In [12]:
import shutil
os.makedirs("data", exist_ok=True)
if os.path.exists("notebooks/data/ds004504"):
    shutil.move("notebooks/data/ds004504", "data/ds004504")
for d in ["data", "results", "report", "src"]:
    p = os.path.join("notebooks", d)
    if os.path.exists(p):
        shutil.rmtree(p)
print(sorted(os.listdir("notebooks")))
print(os.listdir("data/ds004504"))

['.gitkeep', 'audit.ipynb']
['CHANGES', 'dataset_description.json', 'ds004504', 'participants.json', 'participants.tsv', 'README']


In [13]:
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
import pandas as pd
df = pd.read_csv("data/ds004504/participants.tsv", sep="\t")
summary = df.groupby("Group").agg(n=("participant_id", "count"),
                                  age_mean=("Age", "mean"), age_sd=("Age", "std"),
                                  mmse_mean=("MMSE", "mean"), mmse_sd=("MMSE", "std"))
summary.round(1).to_csv("results/tables/cohort_summary.csv")
print(os.path.exists("results/tables/cohort_summary.csv"))

True


In [1]:
import os, urllib.request

base = "https://s3.amazonaws.com/openneuro.org/ds004504/derivatives"
subjects = {"sub-001": "AD", "sub-037": "CN", "sub-066": "FTD"}

for sub, group in subjects.items():
    folder = f"data/ds004504/derivatives/{sub}/eeg"
    os.makedirs(folder, exist_ok=True)
    name = f"{sub}_task-eyesclosed_eeg.set"
    print(f"Downloading {sub} ({group}) ...")
    try:
        urllib.request.urlretrieve(f"{base}/{sub}/eeg/{name}", f"{folder}/{name}")
        size = os.path.getsize(f"{folder}/{name}") / 1e6
        print(f"  done, {size:.1f} MB")
    except Exception as e:
        print("  failed:", e)

  done, 24.4 MB
  done, 31.5 MB
  done, 23.4 MB


In [2]:
import os
for sub in ["sub-001", "sub-037", "sub-066"]:
    folder = f"data/ds004504/derivatives/{sub}/eeg"
    for f in os.listdir(folder):
        print(sub, f, round(os.path.getsize(os.path.join(folder, f)) / 1e6, 1), "MB")

sub-001 sub-001_task-eyesclosed_eeg.set 24.4 MB
sub-037 sub-037_task-eyesclosed_eeg.set 31.5 MB
sub-066 sub-066_task-eyesclosed_eeg.set 23.4 MB


In [3]:
import glob, mne

path = glob.glob("data/ds004504/derivatives/sub-001/eeg/*.set")[0]
raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)
print(raw.info)
print("Duration (s):", raw.times[-1])

<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, Fp2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T3, T4, T5, ...
 chs: 19 EEG
 custom_ref_applied: False
 dig: 22 items (3 Cardinal, 19 EEG)
 highpass: 0.0 Hz
 lowpass: 250.0 Hz
 meas_date: unspecified
 nchan: 19
 projs: []
 sfreq: 500.0 Hz
>
Duration (s): 599.798


In [2]:
import os, socket, urllib.request
import pandas as pd

os.chdir(r"C:\Users\Alive\Documents\eeg-brain-networks-dementia")
socket.setdefaulttimeout(60)

df = pd.read_csv("data/ds004504/participants.tsv", sep="\t")
base = "https://s3.amazonaws.com/openneuro.org/ds004504/derivatives"
failed = []

for i, sub in enumerate(df["participant_id"], 1):
    folder = f"data/ds004504/derivatives/{sub}/eeg"
    name = f"{sub}_task-eyesclosed_eeg.set"
    final = f"{folder}/{name}"
    if os.path.exists(final) and os.path.getsize(final) > 1e6:
        continue
    os.makedirs(folder, exist_ok=True)
    try:
        urllib.request.urlretrieve(f"{base}/{sub}/eeg/{name}", final + ".part")
        os.replace(final + ".part", final)
        print(f"[{i}/88] {sub} done, {os.path.getsize(final)/1e6:.1f} MB")
    except Exception as e:
        failed.append(sub)
        print(f"[{i}/88] {sub} FAILED: {e}")

print("Failed:", failed)

[4/88] sub-004 done, 30.1 MB
[5/88] sub-005 done, 32.6 MB
[6/88] sub-006 done, 26.9 MB
[7/88] sub-007 done, 32.6 MB
[8/88] sub-008 done, 33.8 MB
[9/88] sub-009 done, 26.1 MB
[10/88] sub-010 done, 54.5 MB
[11/88] sub-011 done, 32.8 MB
[12/88] sub-012 done, 37.6 MB
[13/88] sub-013 done, 35.7 MB
[14/88] sub-014 done, 39.8 MB
[15/88] sub-015 done, 38.4 MB
[16/88] sub-016 done, 41.4 MB
[17/88] sub-017 done, 35.9 MB
[18/88] sub-018 done, 36.0 MB
[19/88] sub-019 done, 39.1 MB
[20/88] sub-020 done, 35.3 MB
[21/88] sub-021 done, 39.2 MB
[22/88] sub-022 done, 35.0 MB
[23/88] sub-023 done, 35.6 MB
[24/88] sub-024 done, 32.4 MB
[25/88] sub-025 done, 29.2 MB
[26/88] sub-026 done, 38.3 MB
[27/88] sub-027 done, 35.2 MB
[28/88] sub-028 done, 34.8 MB
[29/88] sub-029 done, 31.4 MB
[30/88] sub-030 done, 23.6 MB
[31/88] sub-031 done, 49.0 MB
[32/88] sub-032 done, 34.0 MB
[33/88] sub-033 done, 30.0 MB
[34/88] sub-034 done, 41.3 MB
[35/88] sub-035 done, 31.6 MB
[36/88] sub-036 done, 35.8 MB
[38/88] sub-038 

In [3]:
import glob
files = glob.glob("data/ds004504/derivatives/*/eeg/*.set")
print(len(files))

88


In [1]:
import os, glob
import numpy as np, pandas as pd, mne
os.chdir(r"C:\Users\Alive\Documents\eeg-brain-networks-dementia")
mne.set_log_level("ERROR")

df = pd.read_csv("data/ds004504/participants.tsv", sep="\t")
rows = []
for k, sub in enumerate(df["participant_id"], 1):
    path = glob.glob(f"data/ds004504/derivatives/{sub}/eeg/*.set")[0]
    raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)
    n_boundary = sum("boundary" in d.lower() for d in raw.annotations.description)
    raw.annotations.description[:] = [
        "BAD_boundary" if "boundary" in d.lower() else d for d in raw.annotations.description]
    ep = mne.make_fixed_length_epochs(raw, duration=4.0, preload=True, verbose=False)
    worst = np.ptp(ep.get_data(), axis=2).max(axis=1) * 1e6      # µV, worst channel per epoch
    for i, w in enumerate(worst):
        rows.append(dict(participant_id=sub, epoch=i, worst_uv=w))
    print(f"[{k}/88] {sub}: {len(worst)} epochs, {n_boundary} boundary marks, "
          f"{raw.n_times / raw.info['sfreq']:.0f} s, {raw.info['nchan']} channels, {raw.info['sfreq']:.0f} Hz")

amp = pd.DataFrame(rows).merge(df[["participant_id", "Group"]], on="participant_id")
amp.to_csv("results/tables/epoch_amplitudes.csv", index=False)

[1/88] sub-001: 149 epochs, 0 boundary marks, 600 s, 19 channels, 500 Hz
[2/88] sub-002: 198 epochs, 0 boundary marks, 793 s, 19 channels, 500 Hz


C:\Users\Alive\AppData\Local\Temp\ipykernel_13780\2534463757.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)


[3/88] sub-003: 75 epochs, 1 boundary marks, 306 s, 19 channels, 500 Hz


C:\Users\Alive\AppData\Local\Temp\ipykernel_13780\2534463757.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)


[4/88] sub-004: 174 epochs, 1 boundary marks, 706 s, 19 channels, 500 Hz
[5/88] sub-005: 201 epochs, 0 boundary marks, 804 s, 19 channels, 500 Hz


...

[86/88] sub-086: 88 epochs, 53 boundary marks, 579 s, 19 channels, 500 Hz


C:\Users\Alive\AppData\Local\Temp\ipykernel_13780\2534463757.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)


[87/88] sub-087: 147 epochs, 3 boundary marks, 603 s, 19 channels, 500 Hz


C:\Users\Alive\AppData\Local\Temp\ipykernel_13780\2534463757.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)


[88/88] sub-088: 188 epochs, 6 boundary marks, 785 s, 19 channels, 500 Hz


In [2]:
res = []
for thr in [150, 200, 250, 300, 350]:
    kept = amp[amp.worst_uv < thr].groupby("participant_id").size()
    k = df.set_index("participant_id").join(kept.rename("kept")).fillna({"kept": 0})
    for g, d in k.groupby("Group"):
        res.append(dict(threshold=thr, Group=g, median_kept=d.kept.median(),
                        min_kept=int(d.kept.min()), n_below_60=int((d.kept < 60).sum())))
print(pd.DataFrame(res).to_string(index=False))

print(amp.groupby("Group").worst_uv.describe(percentiles=[.1, .5, .9]).round(0))

 threshold Group  median_kept  min_kept  n_below_60
       150     A         26.5         0          34
       150     C         20.0         2          27
       150     F         29.0         0          19
       200     A        135.0        10           1
       200     C        139.0        44           1
       200     F        128.0         0           3
       250     A        176.5        65           0
       250     C        185.0       121           0
       250     F        161.0         3           2
       300     A        190.0        75           0
       300     C        193.0       128           0
       300     F        165.0         7           1
       350     A        193.0        75           0
       350     C        197.0       130           0
       350     F        166.0         9           1
        count   mean   std   min    10%    50%    90%     max
Group                                                        
A      7037.0  192.0  66.0  58.0  142.0  179

In [5]:
amp_n = amp.groupby("participant_id").agg(n_epochs=("worst_uv", "size"),
                                          median_uv=("worst_uv", "median"))
for thr in [250, 300]:
    amp_n[f"kept_{thr}"] = amp[amp.worst_uv < thr].groupby("participant_id").size()
amp_n = amp_n.fillna(0).join(df.set_index("participant_id")[["Group"]])
print(amp_n[(amp_n.kept_250 < 60) | (amp_n.kept_300 < 60)].round(0))

per = amp.assign(ok250=amp.worst_uv < 250, ok300=amp.worst_uv < 300)
per = per.groupby(["Group", "participant_id"])[["ok250", "ok300"]].mean()
print(per.groupby("Group").median().round(3))

                n_epochs  median_uv  kept_250  kept_300 Group
participant_id                                               
sub-067              160      261.0        52       138     F
sub-086               88      536.0         3         7     F
       ok250  ok300
Group              
A      0.952  0.979
C      0.940  0.977
F      0.961  0.979
